In [2]:
# --- 必要なパッケージ ---
if (!require(MASS))      install.packages("MASS");      library(MASS)
if (!require(NbClust))   install.packages("NbClust");   library(NbClust)
if (!require(dplyr))     install.packages("dplyr");     library(dplyr)

# --- 設定 ---
filename   <- "preprocessed_features_FP.csv"
index_list <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")
modes      <- c("linear", "nonlinear")
dim_modes  <- c("top3", "cumeig")
target_vars <- c("PCEmax", "Jsc", "Voc", "FF")

# 出力先ディレクトリ＆接尾辞（OH）を一元管理
out_dir <- "outputs_FP"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
suffix <- "FP"  # ファイル名末尾のタグ

# 日付タグ
today_tag <- format(Sys.Date(), "%Y%m%d")

set.seed(42)  # 再現性

# --- データ読み込み ---
readData <- read.delim(filename, header = TRUE, sep = ",",
                       row.names = 1, as.is = TRUE)

# --- 文字列・目的変数除去 ---
expl_vars <- readData[, !(colnames(readData) %in% target_vars), drop = FALSE]
is_char <- sapply(expl_vars, function(col) any(grepl("[A-Za-z]", col) & col != "NA", na.rm = TRUE))
numData <- expl_vars[, !is_char, drop = FALSE]

# --- スケーリング ---
numData_scaled   <- scale(numData)

# ※ 完全相関の削除はスキップ（ご要望どおり）
numData_filtered <- numData_scaled

# --- 相関距離の計算 ---
cor_filtered <- cor(numData_filtered, use = "pairwise.complete.obs")
cor_filtered[is.na(cor_filtered)] <- 0
dist_mat <- 1 - abs(cor_filtered)
dist_mat[dist_mat <= 0] <- 1e-6  # ε補正
ddata <- as.dist(dist_mat)

# --- 線形MDS（固有値あり） ---
n_vars <- ncol(numData_filtered)
cmd_k  <- min(max(1, n_vars - 1), 200)
mds_result_linear <- cmdscale(ddata, eig = TRUE, k = cmd_k)

# --- 固有値寄与率（正の固有値のみ）＆保存 ---
all_eigs <- mds_result_linear$eig
pos_idx  <- which(all_eigs > 0)
pos_eigs <- all_eigs[pos_idx]
contrib  <- if (length(pos_eigs) > 0) pos_eigs / sum(pos_eigs) else numeric(0)
dim_idx  <- seq_along(contrib)

if (length(contrib) > 0) {
  # 折れ線＋累積
  out_pdf_line_full <- file.path(out_dir,
    paste0("mds_linear_eigen_contribution_line_full_", suffix, "_", today_tag, ".pdf")
  )
  pdf(out_pdf_line_full, width = 7, height = 5)
  par(mar = c(4.5, 4.5, 2.5, 1.5))
  plot(dim_idx, contrib, type = "b", pch = 16,
       xlab = "Dimension index", ylab = "Contribution ratio",
       main = "Classical MDS: Eigenvalue Contribution",
       ylim = c(0, max(contrib) * 1.05))
  grid(); lines(dim_idx, cumsum(contrib), lty = 2)
  legend("topright",
         legend = c("Per-dimension contribution", "Cumulative contribution"),
         lty = c(1, 2), pch = c(16, NA), bty = "n")
  dev.off()
  cat("✅ Line plot saved:", out_pdf_line_full, "\n")

  # 棒グラフ（上位50）
  max_dim <- min(50, length(contrib))
  out_pdf_bar_50 <- file.path(out_dir,
    paste0("mds_linear_eigen_contribution_bar_top50_", suffix, "_", today_tag, ".pdf")
  )
  pdf(out_pdf_bar_50, width = 7, height = 5)
  par(mar = c(4.5, 4.5, 2.5, 1.5))
  barplot(contrib[1:max_dim], names.arg = dim_idx[1:max_dim],
          xlab = "Dimension index (up to 50)", ylab = "Contribution ratio",
          main = "Classical MDS: Eigenvalue Contribution (first 50 dims)")
  grid(nx = NA, ny = NULL)
  dev.off()
  cat("✅ Bar plot saved:", out_pdf_bar_50, "\n")
} else {
  warning("No positive eigenvalues found; contribution plots skipped.")
}

# --- cumeig=0.8 に達する次元数を算出（なければ3） ---
k_cumeig <- if (length(contrib) > 0) {
  which(cumsum(contrib) >= 0.8)[1]
} else 3
if (is.na(k_cumeig) || k_cumeig < 2) k_cumeig <- 3

# --- ループ：線形/非線形 × 次元設定 × 指標 ---
for (mode in modes) {
  for (dim_mode in dim_modes) {
    cat("\n=== Processing:", mode, "/", dim_mode, "===\n")

    mds_points <- switch(
      mode,
      "linear" = {
        if (dim_mode == "top3") {
          cmdscale(ddata, eig = FALSE, k = 3)
        } else {
          cmdscale(ddata, eig = FALSE, k = k_cumeig)
        }
      },
      "nonlinear" = {
        MASS::isoMDS(ddata, k = 3)$points
      }
    )

    rownames(mds_points) <- colnames(numData_filtered)

    # 参考：このモードの座標を保存（後で可視化や再利用に便利）
    coord_file <- file.path(out_dir,
      paste0("MDScoords_", mode, "_", dim_mode, "_", suffix, "_", today_tag, ".csv")
    )
    write.csv(mds_points, coord_file)
    cat("💾 Saved coords:", coord_file, "\n")

    for (index in index_list) {
      cat("  └ Index:", index, "\n")
      clustEst <- tryCatch({
        NbClust(
          data = mds_points,
          diss = NULL, distance = "euclidean",
          min.nc = 2, max.nc = 100,
          method = "ward.D2", index = index
        )
      }, error = function(e) {
        warning(paste("Index", index, "failed:", e$message))
        return(NULL)
      })
      if (is.null(clustEst)) next

      best_nc <- clustEst$Best.nc[1]
      grpname <- as.factor(clustEst$Best.partition)
      cat("     Best.nc =", best_nc, "\n")

      # 割当てを表にまとめて保存（変数名＋クラスター＋座標）
      df_cluster <- data.frame(
        Variable = names(grpname),
        Cluster  = grpname,
        stringsAsFactors = FALSE
      )
      for (i_dim in 1:ncol(mds_points)) {
        df_cluster[[paste0("MDS", i_dim)]] <- mds_points[df_cluster$Variable, i_dim]
      }

      # 出力ファイル名：Cluster_{mode}_{dim_mode}_{index}_OH_YYYYMMDD.csv
      fname_csv <- paste0("Cluster_", mode, "_", dim_mode, "_", index, "_",
                          suffix, "_", today_tag, ".csv")
      write.csv(df_cluster, file.path(out_dir, fname_csv), row.names = FALSE)
      cat("     ✅ Saved:", fname_csv, "\n")

      # All.index のプロットを保存：NbClustIndex_{mode}_{dim_mode}_{index}_OH_YYYYMMDD.pdf
      plot_file <- paste0("NbClustIndex_", mode, "_", dim_mode, "_", index, "_",
                          suffix, "_", today_tag, ".pdf")
      pdf(file.path(out_dir, plot_file), width = 7, height = 5)
      plot(clustEst$All.index, type = "b",
           main = paste0("Index: ", index, " (", mode, ", ", dim_mode, ")"),
           xlab = "k", ylab = "Index value")
      grid()
      dev.off()

      # 二次元散布図（1-2軸）も保存：MDS12scatter_{mode}_{dim_mode}_{index}_OH_YYYYMMDD.pdf
      scat_file <- paste0("MDS12scatter_", mode, "_", dim_mode, "_", index, "_",
                          suffix, "_", today_tag, ".pdf")
      pdf(file.path(out_dir, scat_file), width = 6, height = 6)
      xaxis <- 1; yaxis <- 2
      maxiv <- max(c(mds_points[, xaxis], mds_points[, yaxis]))
      miniv <- min(c(mds_points[, xaxis], mds_points[, yaxis]))
      ivdata <- c(as.integer(miniv) - 1, as.integer(maxiv) + 1)
      plot(mds_points[, xaxis], mds_points[, yaxis],
           col = grpname, pch = 19, cex = 0.8,
           xlim = ivdata, ylim = ivdata,
           main = paste0("MDS (", index, ", ", mode, "/", dim_mode, ", k=", best_nc, ")"),
           xlab = paste0("Dim ", xaxis), ylab = paste0("Dim ", yaxis))
      dev.off()
    }
  }
}

cat("\n✅ Completed all: outputs are under ", normalizePath(out_dir), "\n")


Warning message in cor(numData_filtered, use = "pairwise.complete.obs"):
"the standard deviation is zero"
Warning message in cmdscale(ddata, eig = TRUE, k = cmd_k):
"only 135 of the first 200 eigenvalues are > 0"


<U+2705> Line plot saved: outputs_FP/mds_linear_eigen_contribution_line_full_FP_20250821.pdf 
<U+2705> Bar plot saved: outputs_FP/mds_linear_eigen_contribution_bar_top50_FP_20250821.pdf 

=== Processing: linear / top3 ===
<U+0001F4BE> Saved coords: outputs_FP/MDScoords_linear_top3_FP_20250821.csv 
  <U+2514> Index: silhouette 
     Best.nc = 100 
     <U+2705> Saved: Cluster_linear_top3_silhouette_FP_20250821.csv 
  <U+2514> Index: dunn 
     Best.nc = 78 
     <U+2705> Saved: Cluster_linear_top3_dunn_FP_20250821.csv 
  <U+2514> Index: gap 
     Best.nc = 3 
     <U+2705> Saved: Cluster_linear_top3_gap_FP_20250821.csv 
  <U+2514> Index: ch 
     Best.nc = 100 
     <U+2705> Saved: Cluster_linear_top3_ch_FP_20250821.csv 
  <U+2514> Index: db 
     Best.nc = 100 
     <U+2705> Saved: Cluster_linear_top3_db_FP_20250821.csv 
  <U+2514> Index: ptbiserial 
     Best.nc = 4 
     <U+2705> Saved: Cluster_linear_top3_ptbiserial_FP_20250821.csv 

=== Processing: linear / cumeig ===
<U+0001F4BE> 